# Public API Conjugation Walkthrough

This notebook builds an NHS-lysine polymer-protein conjugate through the stable public conjugation API:

```python
from polyzymd.builders.conjugation import build_conjugate_from_config
```

It intentionally does not import internal POC helpers or retired exploratory diagnostics.

The final handoff/export structure from the current public workflow is the solvated PDB exposed as `result.solvated_pdb_path`. The relaxed vacuum-smoke conjugate is exposed separately as `result.relaxed_conjugate_pdb_path`, with explicit minimized/equilibrated smoke paths when available.

In [1]:
import importlib
import sys
from pathlib import Path
from textwrap import dedent


def find_repo_root() -> Path:
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "pyproject.toml").exists() and (base / "src/polyzymd").is_dir():
            return base
    raise FileNotFoundError("Could not locate the PolyzyMD repository root")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

build_conjugate_from_config = importlib.import_module(
    "polyzymd.builders.conjugation"
).build_conjugate_from_config


POC_DIR = REPO_ROOT / "src/polyzymd/builders/conjugation/poc"
SOURCE_PROTEIN_PDB = POC_DIR / "data/NH3_terminal_His_proton_updated.pdb"
OUTPUT_DIR = POC_DIR / "output/public-api-conjugate"
CONFIG_PATH = OUTPUT_DIR / "public_api_conjugation.yaml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Source directory: {SRC_DIR}")
print(f"POC directory: {POC_DIR}")
print(f"Source protein: {SOURCE_PROTEIN_PDB}")
print(f"Output directory: {OUTPUT_DIR}")

Repository root: /home/joelaforet/Shirts-Lab-Linux/polyzymd
Source directory: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src
POC directory: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc
Source protein: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/data/NH3_terminal_His_proton_updated.pdb
Output directory: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate


## Inspect The Protein Fixture

This helper is notebook-local and uses only fixed-column PDB text parsing. It is included only to make the fixture and generated artifacts easy to inspect.

In [2]:
def pdb_summary(path: Path) -> dict[str, object]:
    atom_count = 0
    residues = set()
    chains = set()
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        atom_count += 1
        chain_id = line[21].strip() or "?"
        residue_name = line[17:20].strip()
        residue_number = line[22:26].strip()
        insertion_code = line[26].strip()
        chains.add(chain_id)
        residues.add((chain_id, residue_number, insertion_code, residue_name))
    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "atom_count": atom_count,
        "residue_count": len(residues),
        "chains": sorted(chains),
    }


pdb_summary(SOURCE_PROTEIN_PDB)

{'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/data/NH3_terminal_His_proton_updated.pdb',
 'size_bytes': 280524,
 'atom_count': 2721,
 'residue_count': 181,
 'chains': ['A']}

## Write A Minimal Public Config

The config attaches the validated seeded SBMA/EGPMA/NHS 10-mer reference product to chain `A`, residue `LYS 23`. The shorter fixed `ACB` 3-mer is intentionally not used here because its terminal/reactive residue graph currently cannot be represented by Pablo product-state residue definitions while preserving per-monomer `SBM`/`NHX`/`EGP` identity. The notebook omits free non-covalent polymers so the workflow focuses on the covalent polymer-protein conjugate and final solvation handoff PDB.

In [3]:
CONFIG_TEXT = dedent(
    f"""
    name: public-api-conjugation-walkthrough
    engine: openmm
    enzyme:
      name: public-api-protein
      pdb_path: {SOURCE_PROTEIN_PDB.as_posix()}
    thermodynamics:
      temperature: 300.0
      pressure: 1.0
    simulation_phases:
      equilibration_stages:
        - name: eq
          duration: 0.001
          samples: 1
          ensemble: NVT
          temperature: 300.0
      production:
        ensemble: NVT
        duration: 0.001
        samples: 1
        time_step: 2.0
        report_interval: 1
        checkpoint_interval: 60
    solvent:
      primary:
        type: water
        model: tip3p
      ions:
        neutralize: true
        nacl_concentration: 0.0
      box:
        padding: 0.8
        shape: cube
        tolerance: 2.0
    conjugation:
      enabled: true
      mode: construct
      attachments:
        - name: lys23-reference-sbma-egpma-nhs
          site:
            chain_id: A
            residue_name: LYS
            residue_number: 23
          moiety:
            name: reference-SBMA-EGPMA-NHS
            recipe:
              name: SBMA-EGPMA-NHS
              length: 10
              seed: 42
              reactive_monomer_label: C
              monomers:
                - label: A
                  name: SBMA
                  residue_name: SBM
                  smiles: "[H]C([H])=C(C(=O)OC([H])([H])C([H])([H])[N+](C([H])([H])[H])(C([H])([H])[H])C([H])([H])C([H])([H])C([H])([H])S(=O)(=O)[O-])C([H])([H])[H]"
                  probability: 0.945
                - label: B
                  name: EGPMA
                  residue_name: EGP
                  smiles: "[H]C([H])=C(C(=O)OC([H])([H])C([H])([H])Oc1c([H])c([H])c([H])c([H])c1[H])C([H])([H])[H]"
                  probability: 0.045
                - label: C
                  name: NHS
                  residue_name: NHS
                  smiles: "CC(=C)C(=O)ON1C(=O)CCC1=O"
                  probability: 0.01
          mechanism:
            name: nhs_lys_amide
    """
).strip()

CONFIG_PATH.write_text(CONFIG_TEXT + "\n", encoding="utf-8")
print(f"Wrote config: {CONFIG_PATH}")

Wrote config: /home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/public_api_conjugation.yaml


## Build Through The Public API

This is the only PolyzyMD conjugation call in the notebook. The returned `ConjugationResult` exposes stable public artifact paths while keeping the delegated legacy workflow object out of serialized output.

In [4]:
result = build_conjugate_from_config(
    CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    free_polymer_seed=2026,
)

result.model_dump(exclude={"legacy_result"})

[20:31:35] WARNING: not removing hydrogen atom with dummy atom neighbors


/home/joelaforet/Shirts-Lab-Linux/polyzymd/.pixi/envs/build/lib/python3.12/site-packages/openff/interchange/components/interchange.py:1119: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, functools._lru_cache_wrapper) and obj.__module__.startswith("openff.interchange"):


{'status': 'completed',
 'output_dir': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate'),
 'config_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/public_api_conjugation.yaml'),
 'crosslinked_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/conjugate-construction/assembled_crosslinked.pdb'),
 'minimized_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/conjugate-construction/seeded_random_10mer_crosslinked_relaxed.pdb'),
 'equilibrated_conjugate_pdb_path': None,
 'relaxed_conjugate_pdb_path': PosixPath('/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/conjugate-construction/seeded_random_10mer_cross

## Validate Public Artifacts

The minimized and relaxed conjugate PDBs come from the restrained vacuum smoke stage. The solvated PDB is the handoff/export PDB for later PolyzyMD workflows.

In [5]:
def require_nonempty_path(label: str, path: Path | None) -> Path:
    assert path is not None, f"{label} path was not reported"
    path = Path(path)
    assert path.exists(), f"{label} does not exist: {path}"
    assert path.stat().st_size > 0, f"{label} is empty: {path}"
    return path


validated_paths = {
    "crosslinked_conjugate_pdb": require_nonempty_path(
        "crosslinked conjugate PDB", result.crosslinked_conjugate_pdb_path
    ),
    "minimized_conjugate_pdb": require_nonempty_path(
        "minimized conjugate PDB", result.minimized_conjugate_pdb_path
    ),
    "relaxed_conjugate_pdb": require_nonempty_path(
        "relaxed conjugate PDB", result.relaxed_conjugate_pdb_path
    ),
    "handoff_solvated_pdb": require_nonempty_path(
        "handoff/export solvated PDB", result.solvated_pdb_path
    ),
    "workflow_json": require_nonempty_path(
        "workflow JSON sidecar", result.workflow_json_path
    ),
}

{name: pdb_summary(path) if path.suffix == ".pdb" else str(path) for name, path in validated_paths.items()}

{'crosslinked_conjugate_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/conjugate-construction/assembled_crosslinked.pdb',
  'size_bytes': 257288,
  'atom_count': 3078,
  'residue_count': 191,
  'chains': ['A', 'C']},
 'minimized_conjugate_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/conjugate-construction/seeded_random_10mer_crosslinked_relaxed.pdb',
  'size_bytes': 257288,
  'atom_count': 3078,
  'residue_count': 191,
  'chains': ['A', 'C']},
 'relaxed_conjugate_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/polyzymd/builders/conjugation/poc/output/public-api-conjugate/conjugate-construction/seeded_random_10mer_crosslinked_relaxed.pdb',
  'size_bytes': 257288,
  'atom_count': 3078,
  'residue_count': 191,
  'chains': ['A', 'C']},
 'handoff_solvated_pdb': {'path': '/home/joelaforet/Shirts-Lab-Linux/polyzymd/src/pol

## Handoff

Use `result.solvated_pdb_path` as the exported handoff PDB for later setup, simulation, or analysis workflows. Use `result.relaxed_conjugate_pdb_path` when you need the minimized/relaxed conjugate before solvation.